In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
import sys
import asyncio

# Fix for Windows issues in Jupyter notebooks
if sys.platform == "win32":
    # 1. Use ProactorEventLoop for subprocess support
    if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    
    # 2. Redirect stderr to avoid fileno() error when launching MCP servers
    if "ipykernel" in sys.modules:
        sys.stderr = sys.__stderr__


## Local MCP server

In [1]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "local_server": {
                "transport": "stdio",
                "command": "python",
                "args": ["resources/2.1_mcp_server.py"],
            }
    }
)

In [4]:
# get tools
tools = await client.get_tools()

# get resources
resources = await client.get_resources("local_server")

# get prompts
prompt = await client.get_prompt("local_server", "prompt")
prompt = prompt[0].content

In [5]:
from langchain.agents import create_agent

agent = create_agent(
    model="mistral-small-latest",
    tools=tools,
    system_prompt=prompt
)

In [6]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Tell me about the langchain-mcp-adapters library")]},
    config=config
)

In [7]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='Tell me about the langchain-mcp-adapters library', additional_kwargs={}, response_metadata={}, id='80281e6a-d2d3-4094-8699-88aea2cf12ea'),
              AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '9UjrCAVXV', 'type': 'function', 'function': {'name': 'search_web', 'arguments': '{"query": "langchain-mcp-adapters library"}'}, 'index': 0}]}, response_metadata={'token_usage': {'prompt_tokens': 223, 'total_tokens': 240, 'completion_tokens': 17, 'prompt_tokens_details': {'cached_tokens': 0}}, 'model_name': 'mistral-small-latest', 'model': 'mistral-small-latest', 'finish_reason': 'tool_calls', 'model_provider': 'mistralai'}, id='lc_run--019eed6b-e62c-7dc3-a86d-867689e67722-0', tool_calls=[{'name': 'search_web', 'args': {'query': 'langchain-mcp-adapters library'}, 'id': '9UjrCAVXV', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 223, 'output_tokens': 17, 'total_tokens': 240}),
              ToolMessage(conten

## Online MCP

In [ ]:
# can refer website mcp.so

In [8]:
client = MultiServerMCPClient(
    {
        "time": {
            "transport": "stdio",
            "command": "uvx",
            "args": [
                "mcp-server-time",
                "--local-timezone=America/New_York"
            ]
        }
    }
)

tools = await client.get_tools()

In [9]:
agent = create_agent(
    model="mistral-small-latest",
    tools=tools,
)

In [11]:
question = HumanMessage(content="What time is it?")

response = await agent.ainvoke(
    {"messages": [question]}
)

pprint(response)

{'messages': [HumanMessage(content='What time is it?', additional_kwargs={}, response_metadata={}, id='7acba399-f7d7-4d58-b749-94356339780d'),
              AIMessage(content="I'll get the current time for you. Since you didn't specify a timezone, I'll use the local timezone (America/New_York).", additional_kwargs={'tool_calls': [{'id': 'dKNNf2PvE', 'type': 'function', 'function': {'name': 'get_current_time', 'arguments': '{"timezone": "America/New_York"}'}, 'index': 0}]}, response_metadata={'token_usage': {'prompt_tokens': 324, 'total_tokens': 369, 'completion_tokens': 45, 'prompt_tokens_details': {'cached_tokens': 0}}, 'model_name': 'mistral-small-latest', 'model': 'mistral-small-latest', 'finish_reason': 'tool_calls', 'model_provider': 'mistralai'}, id='lc_run--019eed6d-f0af-7453-9d4d-416aaba96abe-0', tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 'America/New_York'}, 'id': 'dKNNf2PvE', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 324,

In [20]:
client = MultiServerMCPClient({
        

    "kiwi-com-flight-search": {
        "transport": "streamable_http",
      "url": "https://mcp.kiwi.com"
    }


    })

tools = await client.get_tools()


question = HumanMessage(content="Flight timings and prices from PHX to BOM on 2026-07-01")

agent = create_agent(
    model="mistral-small-latest",
    tools=tools,
)

response = await agent.ainvoke(
    {"messages": [question]}
)

pprint(response)

{'messages': [HumanMessage(content='Flight timings and prices from PHX to BOM on 2026-07-01', additional_kwargs={}, response_metadata={}, id='62a083ac-42d9-4900-8dba-b921099c3339'),
              AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '2qjUEhsS1', 'type': 'function', 'function': {'name': 'search-flight', 'arguments': '{"flyFrom": "PHX", "flyTo": "BOM", "departureDate": "01/07/2026", "passengers": {"adults": 1}, "curr": "USD", "locale": "en"}'}, 'index': 0}]}, response_metadata={'token_usage': {'prompt_tokens': 1499, 'total_tokens': 1562, 'completion_tokens': 63, 'prompt_tokens_details': {'cached_tokens': 0}}, 'model_name': 'mistral-small-latest', 'model': 'mistral-small-latest', 'finish_reason': 'tool_calls', 'model_provider': 'mistralai'}, id='lc_run--019eed74-49e0-7d73-86e7-91bd77034dbb-0', tool_calls=[{'name': 'search-flight', 'args': {'flyFrom': 'PHX', 'flyTo': 'BOM', 'departureDate': '01/07/2026', 'passengers': {'adults': 1}, 'curr': 'USD', 'locale': 'en'},

In [23]:
pprint(response['messages'][-1].content)

('Here are the available flights from **Phoenix (PHX)** to **Mumbai (BOM)** on '
 '**July 1, 2026**, sorted by price, duration, and other factors:\n'
 '\n'
 '---\n'
 '\n'
 '### **💰 Cheapest Flights**\n'
 '\n'
 '| **Route** | **Departure & Arrival (Duration)** | **Cabin Class** | **Price '
 '(USD)** | **Book Now** |\n'
 '|-----------|------------------------------------|-----------------|-----------------|--------------|\n'
 '| **PHX → BOM** via HNL, HND, SIN | 01/07 08:10 → 07/03 13:40 (2 days 5h '
 '30m) | Economy | **$1,180** | [Book](https://on.kiwi.com/Ukx6xu) |\n'
 '| **PHX → BOM** via IAH, CDG, NBO | 01/07 09:30 → 07/03 15:30 (2 days 6h 0m) '
 '| Economy | **$1,110** | [Book](https://on.kiwi.com/dqVg3i) |\n'
 '| **PHX → BOM** via LAX, ORY, ATH | 01/07 11:32 → 07/03 23:40 (2 days 12h '
 '8m) | Economy | **$1,083** | [Book](https://on.kiwi.com/u1j7el) |\n'
 '| **PHX → BOM** via LAX, ICN, HAN | 01/07 11:32 → 07/03 23:40 (2 days 12h '
 '8m) | Economy | **$1,090** | [Book](https://on.